In [1]:
import os
import SimpleITK as sitk

def register_image_and_mask(
    fixed_image_path,
    moving_image_path,
    mask_image_path,
    output_registered_image_path,
    output_registered_mask_path,
    learning_rate=2.0,
    min_step=1e-4,
    iterations=200,
    tolerance=1e-8
):
    """
    Registers a moving image to a fixed image and applies the same transformation
    to a segmentation mask (if provided).

    Steps:
      1) Read images & print headers.
      2) Perform a rigid (Euler3D) registration.
      3) Convert the rigid transform to an AffineTransform and refine.
      4) Resample the moving image and mask using the final transform.
    """
    # 1) Read images & print headers
    fixed_image = sitk.ReadImage(fixed_image_path, sitk.sitkFloat32)
    moving_image = sitk.ReadImage(moving_image_path, sitk.sitkFloat32)
    if mask_image_path is not None:
        mask_image = sitk.ReadImage(mask_image_path, sitk.sitkUInt8)
    
    print("\n[INFO] Fixed image header:")
    print("  Origin:", fixed_image.GetOrigin())
    print("  Spacing:", fixed_image.GetSpacing())
    print("  Direction:", fixed_image.GetDirection())
    print("  Size:", fixed_image.GetSize())

    print("\n[INFO] Moving image header:")
    print("  Origin:", moving_image.GetOrigin())
    print("  Spacing:", moving_image.GetSpacing())
    print("  Direction:", moving_image.GetDirection())
    print("  Size:", moving_image.GetSize())
    
    # 2) Rigid Registration (Euler3D)
    initial_transform_rigid = sitk.CenteredTransformInitializer(
        fixed_image,
        moving_image,
        sitk.Euler3DTransform(),
        sitk.CenteredTransformInitializerFilter.MOMENTS
    )
    rigid_registration = sitk.ImageRegistrationMethod()
    rigid_registration.SetMetricAsMeanSquares()
    rigid_registration.SetInterpolator(sitk.sitkLinear)
    rigid_registration.SetOptimizerAsRegularStepGradientDescent(
        learningRate=learning_rate,
        minStep=min_step,
        numberOfIterations=iterations,
        gradientMagnitudeTolerance=tolerance
    )
    rigid_registration.SetOptimizerScalesFromPhysicalShift()
    rigid_registration.SetInitialTransform(initial_transform_rigid, inPlace=False)

    print("\n[INFO] Starting rigid registration...")
    final_transform_rigid = rigid_registration.Execute(fixed_image, moving_image)
    print("[INFO] Rigid transform parameters:", final_transform_rigid)

    # 3) Convert Rigid -> Affine & Refine
    euler_transform = sitk.Euler3DTransform(final_transform_rigid.GetParameters())
    euler_transform.SetCenter(final_transform_rigid.GetFixedParameters())
    affine_transform = sitk.AffineTransform(3)
    affine_transform.SetMatrix(euler_transform.GetMatrix())
    affine_transform.SetTranslation(euler_transform.GetTranslation())
    affine_transform.SetCenter(euler_transform.GetCenter())

    affine_registration = sitk.ImageRegistrationMethod()
    affine_registration.SetMetricAsMeanSquares()
    affine_registration.SetInterpolator(sitk.sitkLinear)
    affine_registration.SetOptimizerAsRegularStepGradientDescent(
        learningRate=learning_rate,
        minStep=min_step,
        numberOfIterations=iterations,
        gradientMagnitudeTolerance=tolerance
    )
    affine_registration.SetOptimizerScalesFromPhysicalShift()
    affine_registration.SetInitialTransform(affine_transform, inPlace=False)

    print("\n[INFO] Starting affine registration...")
    final_transform_affine = affine_registration.Execute(fixed_image, moving_image)
    print("[INFO] Final affine transform parameters:", final_transform_affine)

    # 4) Resample the moving image & mask using the final affine transform
    print("\n[INFO] Resampling the moving image with the final affine transform...")
    resampled_image = sitk.Resample(
        moving_image,
        fixed_image,
        final_transform_affine,
        sitk.sitkLinear,
        0.0,
        moving_image.GetPixelID()
    )
    sitk.WriteImage(resampled_image, output_registered_image_path)
    
    if mask_image_path is not None:
        print("[INFO] Resampling the segmentation mask (nearest neighbor)...")
        resampled_mask = sitk.Resample(
            mask_image,
            fixed_image,
            final_transform_affine,
            sitk.sitkNearestNeighbor,
            0,
            mask_image.GetPixelID()
        )
        sitk.WriteImage(resampled_mask, output_registered_mask_path)
        print("[INFO] Registration complete. Files saved to:")
        print("  Image:", output_registered_image_path)
        print("  Mask:", output_registered_mask_path)
    else:
        print("[INFO] Registration complete. Registered image saved to:", output_registered_image_path)

In [6]:
import os
import re

# Set the base directory for the Imaging dataset
base_dir = '/Users/yifanli/Desktop/dataset/GBM_public/Imaging_pefect'
# base_dir ='/Volumes/ssd/dataset/GBM_public/Imaging_perfect'

# List of required filenames
required_files = ['CT1.nii.gz', 'FLAIR.nii.gz', 'T2.nii.gz', 'T1.nii.gz']

# Function to extract the numeric part from a patient folder name like "Patient-001"
def extract_patient_number(patient_name):
    match = re.search(r'Patient-(\d+)', patient_name)
    return int(match.group(1)) if match else float('inf')

# Get sorted patient directories based on their numeric identifier
patient_dirs = sorted(
    [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))],
    key=extract_patient_number
)

# List to store patients that follow the requirement
patients_meeting_requirements = []

# Loop through each patient folder
for patient in patient_dirs:
    patient_path = os.path.join(base_dir, patient)
    
    # Look for either "week-000-1" or "week-000" folder
    week_folder = None
    for candidate in ['week-000-1', 'week-000']:
        candidate_path = os.path.join(patient_path, candidate)
        if os.path.isdir(candidate_path):
            week_folder = candidate
            break
    
    # If neither week folder is found, skip this patient
    if week_folder is None:
        continue
    
    # Check if all required files are present in the found week folder
    week_path = os.path.join(patient_path, week_folder)
    missing_files = []
    for filename in required_files:
        file_path = os.path.join(week_path, filename)
        if not os.path.exists(file_path):
            missing_files.append(filename)
    
    # Only record the patient if no files are missing
    if not missing_files:
        patients_meeting_requirements.append(patient)

# Output the list of patients that meet the requirement
print("Patients with a valid week folder and all required files:")
for patient in patients_meeting_requirements:
    print(patient)


Patients with a valid week folder and all required files:
Patient-004
Patient-010
Patient-013
Patient-014
Patient-015
Patient-022
Patient-023
Patient-025
Patient-028
Patient-029
Patient-030
Patient-032
Patient-033
Patient-034
Patient-037
Patient-038
Patient-040
Patient-042
Patient-045
Patient-048
Patient-049
Patient-051
Patient-057
Patient-059
Patient-060
Patient-066
Patient-068
Patient-071
Patient-072
Patient-077
Patient-078
Patient-081
Patient-086
Patient-091


In [5]:
import os
import nibabel as nib
import re

# Set the base directory for the Imaging dataset
base_dir = '/Users/yifanli/Desktop/dataset/GBM_public/Imaging_pefect'

# Function to extract the numeric part from a patient folder name like "Patient-001"
def extract_patient_number(patient_name):
    match = re.search(r'Patient-(\d+)', patient_name)
    return int(match.group(1)) if match else float('inf')

# Get sorted patient directories based on their numeric identifier
patient_dirs = sorted(
    [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))],
    key=extract_patient_number
)

# Loop over each patient folder
for patient in patient_dirs:
    patient_path = os.path.join(base_dir, patient)
    
    # Get all week folders inside the patient folder (sorted lexicographically)
    week_dirs = sorted(
        [d for d in os.listdir(patient_path) if os.path.isdir(os.path.join(patient_path, d))]
    )
    
    # Process each week folder for the patient
    for week in week_dirs:
        week_path = os.path.join(patient_path, week)
        found = False  # Flag to indicate if any file meets the criteria in this week folder

        # Define the file paths to check
        file_paths = {
            "T2.nii": os.path.join(week_path, "T2.nii.gz"),
            "t2_seg_mask.nii.gz": os.path.join(week_path, "DeepBraTumIA-segmentation", "native", "segmentation", "t2_seg_mask.nii.gz"),
            "segmentation_T2_origspace.nii.gz": os.path.join(week_path, "HD-GLIO-AUTO-segmentation", "native", "segmentation_T2_origspace.nii.gz")
        }
        
        # For each file, attempt to load and check its shape
        for label, path in file_paths.items():
            if os.path.exists(path):
                try:
                    img = nib.load(path)
                    shape = img.shape
                    # Check if the image has 3 dimensions and all dimensions are > 100
                    if len(shape) == 3 and shape[0] > 100 and shape[1] > 100 and shape[2] > 100:
                        print(f"Patient: {patient}, Week: {week} -> {label}: shape = {shape} (all dimensions > 100)")
                        found = True
                except Exception as e:
                    print(f"Error loading {label} for Patient: {patient}, Week: {week} ({e})")
            else:
                print(f"File not found: {path}")
        
        if found:
            print("-" * 60)


Patient: Patient-004, Week: week-000-1 -> T2.nii: shape = (256, 256, 160) (all dimensions > 100)
Patient: Patient-004, Week: week-000-1 -> t2_seg_mask.nii.gz: shape = (256, 256, 160) (all dimensions > 100)
Patient: Patient-004, Week: week-000-1 -> segmentation_T2_origspace.nii.gz: shape = (256, 256, 160) (all dimensions > 100)
------------------------------------------------------------
Patient: Patient-004, Week: week-000-2 -> T2.nii: shape = (256, 256, 160) (all dimensions > 100)
Patient: Patient-004, Week: week-000-2 -> t2_seg_mask.nii.gz: shape = (256, 256, 160) (all dimensions > 100)
Patient: Patient-004, Week: week-000-2 -> segmentation_T2_origspace.nii.gz: shape = (256, 256, 160) (all dimensions > 100)
------------------------------------------------------------
Patient: Patient-004, Week: week-020 -> T2.nii: shape = (256, 256, 160) (all dimensions > 100)
Patient: Patient-004, Week: week-020 -> t2_seg_mask.nii.gz: shape = (256, 256, 160) (all dimensions > 100)
Patient: Patient-0

Patient: Patient-029, Week: week-059 -> t2_seg_mask.nii.gz: shape = (256, 256, 160) (all dimensions > 100)
Patient: Patient-029, Week: week-059 -> segmentation_T2_origspace.nii.gz: shape = (256, 256, 160) (all dimensions > 100)
------------------------------------------------------------
Patient: Patient-029, Week: week-100 -> T2.nii: shape = (256, 256, 176) (all dimensions > 100)
Patient: Patient-029, Week: week-100 -> t2_seg_mask.nii.gz: shape = (256, 256, 176) (all dimensions > 100)
Patient: Patient-029, Week: week-100 -> segmentation_T2_origspace.nii.gz: shape = (256, 256, 176) (all dimensions > 100)
------------------------------------------------------------
Patient: Patient-029, Week: week-112 -> T2.nii: shape = (256, 256, 176) (all dimensions > 100)
Patient: Patient-029, Week: week-112 -> t2_seg_mask.nii.gz: shape = (256, 256, 176) (all dimensions > 100)
Patient: Patient-029, Week: week-112 -> segmentation_T2_origspace.nii.gz: shape = (256, 256, 176) (all dimensions > 100)
----

Patient: Patient-048, Week: week-000-1 -> T2.nii: shape = (256, 256, 160) (all dimensions > 100)
Patient: Patient-048, Week: week-000-1 -> t2_seg_mask.nii.gz: shape = (256, 256, 160) (all dimensions > 100)
Patient: Patient-048, Week: week-000-1 -> segmentation_T2_origspace.nii.gz: shape = (256, 256, 160) (all dimensions > 100)
------------------------------------------------------------
Patient: Patient-048, Week: week-000-2 -> T2.nii: shape = (256, 256, 176) (all dimensions > 100)
Patient: Patient-048, Week: week-000-2 -> t2_seg_mask.nii.gz: shape = (256, 256, 176) (all dimensions > 100)
Patient: Patient-048, Week: week-000-2 -> segmentation_T2_origspace.nii.gz: shape = (256, 256, 176) (all dimensions > 100)
------------------------------------------------------------
Patient: Patient-048, Week: week-013 -> T2.nii: shape = (256, 256, 176) (all dimensions > 100)
Patient: Patient-048, Week: week-013 -> t2_seg_mask.nii.gz: shape = (256, 256, 176) (all dimensions > 100)
Patient: Patient-0

Patient: Patient-072, Week: week-064 -> T2.nii: shape = (256, 256, 160) (all dimensions > 100)
Patient: Patient-072, Week: week-064 -> t2_seg_mask.nii.gz: shape = (256, 256, 160) (all dimensions > 100)
Patient: Patient-072, Week: week-064 -> segmentation_T2_origspace.nii.gz: shape = (256, 256, 160) (all dimensions > 100)
------------------------------------------------------------
Patient: Patient-072, Week: week-073 -> T2.nii: shape = (256, 256, 160) (all dimensions > 100)
Patient: Patient-072, Week: week-073 -> t2_seg_mask.nii.gz: shape = (256, 256, 160) (all dimensions > 100)
Patient: Patient-072, Week: week-073 -> segmentation_T2_origspace.nii.gz: shape = (256, 256, 160) (all dimensions > 100)
------------------------------------------------------------
Patient: Patient-072, Week: week-078 -> T2.nii: shape = (256, 256, 160) (all dimensions > 100)
Patient: Patient-072, Week: week-078 -> t2_seg_mask.nii.gz: shape = (256, 256, 160) (all dimensions > 100)
Patient: Patient-072, Week: we

In [17]:
import os

import nibabel as nib
import re
def register_patient_one_week(
    fixed_image_path,
    week_folder,
    week_T2_mask_path,
    output_dir,
    learning_rate=2.0,
    min_step=1e-4,
    iterations=200,
    tolerance=1e-8
):
    """
    Registers a set of imaging modalities for a single patient for one time point (week).

    It is assumed that the week folder contains the following files (with these fixed names):
      - T1 image:      'T1.nii.gz'
      - CT1 image:     'CT1.nii.gz'
      - T2 image:      'T2.nii.gz'
      - FLAIR image:   'FLAIR.nii.gz'
    
    In addition, one T2 segmentation mask path is provided for the week.
    
    Registered images (and the T2 mask when applicable) are saved to output_dir.
    """
    os.makedirs(output_dir, exist_ok=True)

    # Define modalities and their expected file names.
    modalities = {
        "T1": "T1.nii.gz",
        "CT1": "CT1.nii.gz",
        "T2": "T2.nii.gz",
        "FLAIR": "FLAIR.nii.gz"
    }
    
    # Construct modality paths for the week.
    week_paths = {mod: os.path.join(week_folder, filename) for mod, filename in modalities.items()}
    
    # Add the T2 segmentation mask path.
    week_paths["T2_mask"] = week_T2_mask_path

    # Process each modality.
    for modality in ["T1", "CT1", "T2", "FLAIR"]:
#     for modality in [ "T2"]:
        moving_image_path = week_paths[modality]
        # Only T2 has an associated mask.
        mask_image_path = week_paths["T2_mask"] if modality == "T2" else None
        output_registered_image_path = os.path.join(output_dir, f"registered_{modality}.nii.gz")
        if mask_image_path is not None:
            output_registered_mask_path = os.path.join(output_dir, f"registered_{modality}_mask.nii.gz")
        else:
            output_registered_mask_path = None

        print(f"\n[INFO] Registering {modality} for the week...")
        register_image_and_mask(
            fixed_image_path,
            moving_image_path,
            mask_image_path,
            output_registered_image_path,
            output_registered_mask_path,
            learning_rate,
            min_step,
            iterations,
            tolerance
        )

    print("\n[INFO] All registrations for the week are complete.")


In [26]:
import os
import re

def extract_patient_number(patient_name):
    """
    Extracts the numeric identifier from a patient folder name like "Patient-001".
    Returns an integer or a large number if not found.
    """
    match = re.search(r'Patient-(\d+)', patient_name)
    return int(match.group(1)) if match else float('inf')

# Base directory where your Imaging dataset is stored.
base_dir = '/Users/yifanli/Desktop/dataset/GBM_public/imaging_perfect'
# base_dir = '/Users/yifanli/Desktop/dataset/GBM_public/imaging_trainOnly'
# List of week folder names to include.
matching_weeks = ['week-000', 'week-000-1']

# List to store the matching week folder paths.
matching_week_paths = []

# Get sorted patient directories based on their numeric identifier.
patients = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
patients_sorted = sorted(patients, key=extract_patient_number)

# Loop over each sorted patient folder.
for patient in patients_sorted:
    patient_path = os.path.join(base_dir, patient)
    # Loop over week folders inside the patient folder.
    for week in os.listdir(patient_path):
        if week in matching_weeks:
            week_path = os.path.join(patient_path, week)
            if os.path.isdir(week_path):
                matching_week_paths.append(week_path)

# Print out the list of matching week folder paths, ordered by patient number.
print("Matching week folder paths (ordered by patient number):")
for path in matching_week_paths:
    print(path)


Matching week folder paths (ordered by patient number):
/Users/yifanli/Desktop/dataset/GBM_public/imaging_perfect/Patient-004/week-000-1
/Users/yifanli/Desktop/dataset/GBM_public/imaging_perfect/Patient-010/week-000
/Users/yifanli/Desktop/dataset/GBM_public/imaging_perfect/Patient-013/week-000
/Users/yifanli/Desktop/dataset/GBM_public/imaging_perfect/Patient-014/week-000
/Users/yifanli/Desktop/dataset/GBM_public/imaging_perfect/Patient-015/week-000-1
/Users/yifanli/Desktop/dataset/GBM_public/imaging_perfect/Patient-022/week-000
/Users/yifanli/Desktop/dataset/GBM_public/imaging_perfect/Patient-023/week-000
/Users/yifanli/Desktop/dataset/GBM_public/imaging_perfect/Patient-025/week-000
/Users/yifanli/Desktop/dataset/GBM_public/imaging_perfect/Patient-028/week-000-1
/Users/yifanli/Desktop/dataset/GBM_public/imaging_perfect/Patient-029/week-000-1
/Users/yifanli/Desktop/dataset/GBM_public/imaging_perfect/Patient-030/week-000
/Users/yifanli/Desktop/dataset/GBM_public/imaging_perfect/Patient-0

In [27]:
import tqdm
fixed_image_path = '/Users/yifanli/Downloads/sri24_spm8/templates/T2_brain.nii'

# Base output directory where the processed results will be saved.
output_base = '/Users/yifanli/Desktop/dataset/GBM_public/procesed_perfect'

# Loop over each matching week folder
for week_folder in tqdm.tqdm(matching_week_paths):
    # Extract patient name from the path; assumes folder structure .../Patient-XXX/week-000*
    patient_name = os.path.basename(os.path.dirname(week_folder))
    # Create a dedicated output folder for the patient
    output_dir = os.path.join(output_base, patient_name)
    os.makedirs(output_dir, exist_ok=True)
    
    # Determine the T2 mask path.
    # Option 2: HD-GLIO-AUTO-segmentation native mask
    mask_option2 = os.path.join(week_folder, "HD-GLIO-AUTO-segmentation", "native", "segmentation_T2_origspace.nii.gz")
    # Option 1: DeepBraTumIA segmentation mask
    mask_option1 = os.path.join(week_folder, "DeepBraTumIA-segmentation", "native", "segmentation", "t2_seg_mask.nii.gz")
    
    if os.path.exists(mask_option1):
        week_T2_mask_path = mask_option1
    elif os.path.exists(mask_option2):
        week_T2_mask_path = mask_option2
    else:
        print(f"No T2 mask found for {week_folder}. Skipping registration for this folder.")
        continue
    
    print(f"Registering patient {patient_name} for week folder: {week_folder}")
    
    # Call the registration function for one week.
    register_patient_one_week(
        fixed_image_path,
        week_folder,
        week_T2_mask_path,
        output_dir
    )

  0%|                                                    | 0/34 [00:00<?, ?it/s]

Registering patient Patient-004 for week folder: /Users/yifanli/Desktop/dataset/GBM_public/imaging_perfect/Patient-004/week-000-1

[INFO] Registering T2 for the week...

[INFO] Fixed image header:
  Origin: (120.0, 129.0, -68.0)
  Spacing: (1.0, 1.0, 1.0)
  Direction: (-1.0, 0.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0, 1.0)
  Size: (240, 240, 155)

[INFO] Moving image header:
  Origin: (78.42269134521484, -131.47691345214844, 110.97135925292969)
  Spacing: (1.0, 1.0, 1.0)
  Direction: (7.614516900946548e-09, 7.615845615842184e-09, -1.0, 0.9999999847696381, 0.00017453000789585297, 7.61584573183427e-09, 0.00017453000789585297, -0.9999999847696381, -7.614517016918398e-09)
  Size: (256, 256, 160)

[INFO] Starting rigid registration...
[INFO] Rigid transform parameters: itk::simple::Transform
 CompositeTransform (0x7f93ab118dd0)
   RTTI typeinfo:   itk::CompositeTransform<double, 3u>
   Reference Count: 1
   Modified Time: 701045
   Debug: Off
   Object Name: 
   Observers: 
     none
   Transforms 

  3%|█▏                                       | 1/34 [02:19<1:16:44, 139.54s/it]

[INFO] Resampling the segmentation mask (nearest neighbor)...
[INFO] Registration complete. Files saved to:
  Image: /Users/yifanli/Desktop/dataset/GBM_public/procesed_perfect/Patient-004/registered_T2.nii.gz
  Mask: /Users/yifanli/Desktop/dataset/GBM_public/procesed_perfect/Patient-004/registered_T2_mask.nii.gz

[INFO] All registrations for the week are complete.
Registering patient Patient-010 for week folder: /Users/yifanli/Desktop/dataset/GBM_public/imaging_perfect/Patient-010/week-000

[INFO] Registering T2 for the week...

[INFO] Fixed image header:
  Origin: (120.0, 129.0, -68.0)
  Spacing: (1.0, 1.0, 1.0)
  Direction: (-1.0, 0.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0, 1.0)
  Size: (240, 240, 155)

[INFO] Moving image header:
  Origin: (87.49954223632812, -130.98089599609375, 107.82530212402344)
  Spacing: (1.00390625, 1.00390625, 1.0)
  Direction: (-0.0, 0.0, -1.0, 1.0, -0.0, 0.0, 0.0, -1.0, 0.0)
  Size: (256, 256, 176)

[INFO] Starting rigid registration...
[INFO] Rigid transform para

  6%|██▍                                      | 2/34 [05:12<1:25:02, 159.45s/it]

...

[INFO] Resampling the segmentation mask (nearest neighbor)...
[INFO] Registration complete. Files saved to:
  Image: /Users/yifanli/Desktop/dataset/GBM_public/procesed_perfect/Patient-081/registered_T2.nii.gz
  Mask: /Users/yifanli/Desktop/dataset/GBM_public/procesed_perfect/Patient-081/registered_T2_mask.nii.gz

[INFO] All registrations for the week are complete.
Registering patient Patient-086 for week folder: /Users/yifanli/Desktop/dataset/GBM_public/imaging_perfect/Patient-086/week-000

[INFO] Registering T2 for the week...

[INFO] Fixed image header:
  Origin: (120.0, 129.0, -68.0)
  Spacing: (1.0, 1.0, 1.0)
  Direction: (-1.0, 0.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0, 1.0)
  Size: (240, 240, 155)

[INFO] Moving image header:
  Origin: (-108.2279052734375, -105.17736053466797, -31.239665985107422)
  Spacing: (0.6875, 0.6875, 5.999998569488525)
  Direction: (0.9995558648646377, 0.012211269802824646, 0.027183780928667908, -0.018102060972711515, 0.9734105944104601, 0.2283509193139844, -0.0

 97%|██████████████████████████████████████▊ | 33/34 [1:50:56<03:21, 201.49s/it]

[INFO] Resampling the segmentation mask (nearest neighbor)...
[INFO] Registration complete. Files saved to:
  Image: /Users/yifanli/Desktop/dataset/GBM_public/procesed_perfect/Patient-086/registered_T2.nii.gz
  Mask: /Users/yifanli/Desktop/dataset/GBM_public/procesed_perfect/Patient-086/registered_T2_mask.nii.gz

[INFO] All registrations for the week are complete.
Registering patient Patient-091 for week folder: /Users/yifanli/Desktop/dataset/GBM_public/imaging_perfect/Patient-091/week-000

[INFO] Registering T2 for the week...

[INFO] Fixed image header:
  Origin: (120.0, 129.0, -68.0)
  Spacing: (1.0, 1.0, 1.0)
  Direction: (-1.0, 0.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0, 1.0)
  Size: (240, 240, 155)

[INFO] Moving image header:
  Origin: (81.05193328857422, -129.1395721435547, 106.76077270507812)
  Spacing: (0.5, 0.5, 1.0)
  Direction: (-0.0, 0.0, -1.0, 1.0, -0.0, 0.0, 0.0, -1.0, 0.0)
  Size: (512, 512, 176)

[INFO] Starting rigid registration...
[INFO] Rigid transform parameters: itk::si

100%|████████████████████████████████████████| 34/34 [1:54:44<00:00, 202.48s/it]

[INFO] Resampling the segmentation mask (nearest neighbor)...
[INFO] Registration complete. Files saved to:
  Image: /Users/yifanli/Desktop/dataset/GBM_public/procesed_perfect/Patient-091/registered_T2.nii.gz
  Mask: /Users/yifanli/Desktop/dataset/GBM_public/procesed_perfect/Patient-091/registered_T2_mask.nii.gz

[INFO] All registrations for the week are complete.


In [28]:
# Base directory where your Imaging dataset is stored.

base_dir = '/Users/yifanli/Desktop/dataset/GBM_public/imaging_trainOnly'
# List of week folder names to include.
matching_weeks = ['week-000', 'week-000-1']

# List to store the matching week folder paths.
matching_week_paths = []

# Get sorted patient directories based on their numeric identifier.
patients = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
patients_sorted = sorted(patients, key=extract_patient_number)

# Loop over each sorted patient folder.
for patient in patients_sorted:
    patient_path = os.path.join(base_dir, patient)
    # Loop over week folders inside the patient folder.
    for week in os.listdir(patient_path):
        if week in matching_weeks:
            week_path = os.path.join(patient_path, week)
            if os.path.isdir(week_path):
                matching_week_paths.append(week_path)

# Print out the list of matching week folder paths, ordered by patient number.
print("Matching week folder paths (ordered by patient number):")
for path in matching_week_paths:
    print(path)


Matching week folder paths (ordered by patient number):
/Users/yifanli/Desktop/dataset/GBM_public/imaging_trainOnly/Patient-012/week-000-1
/Users/yifanli/Desktop/dataset/GBM_public/imaging_trainOnly/Patient-018/week-000
/Users/yifanli/Desktop/dataset/GBM_public/imaging_trainOnly/Patient-020/week-000-1
/Users/yifanli/Desktop/dataset/GBM_public/imaging_trainOnly/Patient-031/week-000
/Users/yifanli/Desktop/dataset/GBM_public/imaging_trainOnly/Patient-043/week-000
/Users/yifanli/Desktop/dataset/GBM_public/imaging_trainOnly/Patient-052/week-000
/Users/yifanli/Desktop/dataset/GBM_public/imaging_trainOnly/Patient-055/week-000-1
/Users/yifanli/Desktop/dataset/GBM_public/imaging_trainOnly/Patient-061/week-000-1
/Users/yifanli/Desktop/dataset/GBM_public/imaging_trainOnly/Patient-062/week-000-1
/Users/yifanli/Desktop/dataset/GBM_public/imaging_trainOnly/Patient-067/week-000-1
/Users/yifanli/Desktop/dataset/GBM_public/imaging_trainOnly/Patient-069/week-000
/Users/yifanli/Desktop/dataset/GBM_public

In [29]:
import tqdm
fixed_image_path = '/Users/yifanli/Downloads/sri24_spm8/templates/T2_brain.nii'

# Base output directory where the processed results will be saved.
output_base = '/Users/yifanli/Desktop/dataset/GBM_public/procesed_trainOnly'

# Loop over each matching week folder
for week_folder in tqdm.tqdm(matching_week_paths):
    # Extract patient name from the path; assumes folder structure .../Patient-XXX/week-000*
    patient_name = os.path.basename(os.path.dirname(week_folder))
    # Create a dedicated output folder for the patient
    output_dir = os.path.join(output_base, patient_name)
    os.makedirs(output_dir, exist_ok=True)
    
    # Determine the T2 mask path.
    # Option 1: HD-GLIO-AUTO-segmentation native mask
    mask_option2 = os.path.join(week_folder, "HD-GLIO-AUTO-segmentation", "native", "segmentation_T2_origspace.nii.gz")
    # Option 2: DeepBraTumIA segmentation mask
    mask_option1 = os.path.join(week_folder, "DeepBraTumIA-segmentation", "native", "segmentation", "t2_seg_mask.nii.gz")
    
    if os.path.exists(mask_option1):
        week_T2_mask_path = mask_option1
    elif os.path.exists(mask_option2):
        week_T2_mask_path = mask_option2
    else:
        print(f"No T2 mask found for {week_folder}. Skipping registration for this folder.")
        continue
    
    print(f"Registering patient {patient_name} for week folder: {week_folder}")
    
    # Call the registration function for one week.
    register_patient_one_week(
        fixed_image_path,
        week_folder,
        week_T2_mask_path,
        output_dir
    )

  0%|                                                    | 0/14 [00:00<?, ?it/s]

Registering patient Patient-012 for week folder: /Users/yifanli/Desktop/dataset/GBM_public/imaging_trainOnly/Patient-012/week-000-1

[INFO] Registering T2 for the week...

[INFO] Fixed image header:
  Origin: (120.0, 129.0, -68.0)
  Spacing: (1.0, 1.0, 1.0)
  Direction: (-1.0, 0.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0, 1.0)
  Size: (240, 240, 155)

[INFO] Moving image header:
  Origin: (-115.53314971923828, -108.84062194824219, -47.549652099609375)
  Spacing: (0.4296875, 0.4296875, 5.999998092651367)
  Direction: (0.9957262809168557, 0.0661338052035931, -0.06446312451507319, -0.04223665716470459, 0.9468268408510357, 0.31895925911480544, 0.08212939962667726, -0.3148733924188269, 0.9455736336228335)
  Size: (448, 512, 24)

[INFO] Starting rigid registration...
[INFO] Rigid transform parameters: itk::simple::Transform
 CompositeTransform (0x7f939b8b6220)
   RTTI typeinfo:   itk::CompositeTransform<double, 3u>
   Reference Count: 1
   Modified Time: 836571
   Debug: Off
   Object Name: 
   Observ

  7%|███                                        | 1/14 [02:57<38:30, 177.73s/it]

[INFO] Resampling the segmentation mask (nearest neighbor)...
[INFO] Registration complete. Files saved to:
  Image: /Users/yifanli/Desktop/dataset/GBM_public/procesed_trainOnly/Patient-012/registered_T2.nii.gz
  Mask: /Users/yifanli/Desktop/dataset/GBM_public/procesed_trainOnly/Patient-012/registered_T2_mask.nii.gz

[INFO] All registrations for the week are complete.
Registering patient Patient-018 for week folder: /Users/yifanli/Desktop/dataset/GBM_public/imaging_trainOnly/Patient-018/week-000

[INFO] Registering T2 for the week...

[INFO] Fixed image header:
  Origin: (120.0, 129.0, -68.0)
  Spacing: (1.0, 1.0, 1.0)
  Direction: (-1.0, 0.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0, 1.0)
  Size: (240, 240, 155)

[INFO] Moving image header:
  Origin: (-93.27206420898438, -141.30950927734375, 7.1025800704956055)
  Spacing: (0.4296875, 0.4296875, 6.000003337860107)
  Direction: (0.9905775498993826, 0.07621909297534508, -0.11378386212290939, -0.047879925199445485, 0.9711318585366435, 0.233688720475

 14%|██████▏                                    | 2/14 [05:50<35:01, 175.08s/it]

...

[INFO] Resampling the segmentation mask (nearest neighbor)...
[INFO] Registration complete. Files saved to:
  Image: /Users/yifanli/Desktop/dataset/GBM_public/procesed_trainOnly/Patient-073/registered_T2.nii.gz
  Mask: /Users/yifanli/Desktop/dataset/GBM_public/procesed_trainOnly/Patient-073/registered_T2_mask.nii.gz

[INFO] All registrations for the week are complete.
Registering patient Patient-075 for week folder: /Users/yifanli/Desktop/dataset/GBM_public/imaging_trainOnly/Patient-075/week-000

[INFO] Registering T2 for the week...

[INFO] Fixed image header:
  Origin: (120.0, 129.0, -68.0)
  Spacing: (1.0, 1.0, 1.0)
  Direction: (-1.0, 0.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0, 1.0)
  Size: (240, 240, 155)

[INFO] Moving image header:
  Origin: (-97.87333679199219, -129.72756958007812, -31.82225799560547)
  Spacing: (0.4910714030265808, 0.4910714030265808, 5.999998092651367)
  Direction: (0.9995427718121684, 1.710960046055045e-05, -0.03023651609762089, 0.00816928595511372, 0.96265702305108

 93%|███████████████████████████████████████   | 13/14 [36:45<02:49, 169.50s/it]

[INFO] Resampling the segmentation mask (nearest neighbor)...
[INFO] Registration complete. Files saved to:
  Image: /Users/yifanli/Desktop/dataset/GBM_public/procesed_trainOnly/Patient-075/registered_T2.nii.gz
  Mask: /Users/yifanli/Desktop/dataset/GBM_public/procesed_trainOnly/Patient-075/registered_T2_mask.nii.gz

[INFO] All registrations for the week are complete.
Registering patient Patient-082 for week folder: /Users/yifanli/Desktop/dataset/GBM_public/imaging_trainOnly/Patient-082/week-000-1

[INFO] Registering T2 for the week...

[INFO] Fixed image header:
  Origin: (120.0, 129.0, -68.0)
  Spacing: (1.0, 1.0, 1.0)
  Direction: (-1.0, 0.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0, 1.0)
  Size: (240, 240, 155)

[INFO] Moving image header:
  Origin: (86.87349700927734, -138.5240936279297, 128.5)
  Spacing: (1.00390625, 1.00390625, 1.0)
  Direction: (-0.0, 0.0, -1.0, 1.0, -0.0, 0.0, 0.0, -1.0, 0.0)
  Size: (256, 256, 176)

[INFO] Starting rigid registration...
[INFO] Rigid transform parameters

100%|██████████████████████████████████████████| 14/14 [40:32<00:00, 173.74s/it]

[INFO] Resampling the segmentation mask (nearest neighbor)...
[INFO] Registration complete. Files saved to:
  Image: /Users/yifanli/Desktop/dataset/GBM_public/procesed_trainOnly/Patient-082/registered_T2.nii.gz
  Mask: /Users/yifanli/Desktop/dataset/GBM_public/procesed_trainOnly/Patient-082/registered_T2_mask.nii.gz

[INFO] All registrations for the week are complete.


In [15]:
import os

base_path = '/Users/yifanli/Desktop/dataset/GBM_public/perfect_week_N'
week_paths = []

# Loop through each patient folder in the base directory
for patient in os.listdir(base_path):
    patient_dir = os.path.join(base_path, patient)
    if os.path.isdir(patient_dir):
        # Loop through items in each patient folder; assuming only one week folder exists per patient
        for item in os.listdir(patient_dir):
            week_folder = os.path.join(patient_dir, item)
            if os.path.isdir(week_folder):
                week_paths.append(week_folder)

print(len(week_paths))
week_paths

31


['/Users/yifanli/Desktop/dataset/GBM_public/perfect_week_N/Patient-037/week-037',
 '/Users/yifanli/Desktop/dataset/GBM_public/perfect_week_N/Patient-030/week-085',
 '/Users/yifanli/Desktop/dataset/GBM_public/perfect_week_N/Patient-091/week-043',
 '/Users/yifanli/Desktop/dataset/GBM_public/perfect_week_N/Patient-038/week-023',
 '/Users/yifanli/Desktop/dataset/GBM_public/perfect_week_N/Patient-025/week-070',
 '/Users/yifanli/Desktop/dataset/GBM_public/perfect_week_N/Patient-022/week-037',
 '/Users/yifanli/Desktop/dataset/GBM_public/perfect_week_N/Patient-014/week-012',
 '/Users/yifanli/Desktop/dataset/GBM_public/perfect_week_N/Patient-013/week-030',
 '/Users/yifanli/Desktop/dataset/GBM_public/perfect_week_N/Patient-078/week-088',
 '/Users/yifanli/Desktop/dataset/GBM_public/perfect_week_N/Patient-040/week-028',
 '/Users/yifanli/Desktop/dataset/GBM_public/perfect_week_N/Patient-049/week-046',
 '/Users/yifanli/Desktop/dataset/GBM_public/perfect_week_N/Patient-071/week-062',
 '/Users/yifanli

In [18]:
import tqdm
fixed_image_path = '/Users/yifanli/Downloads/sri24_spm8/templates/T2_brain.nii'

# Base output directory where the processed results will be saved.
output_base = '/Users/yifanli/Desktop/dataset/GBM_public/procesed_perfect_week_N'

# Loop over each matching week folder
for week_folder in tqdm.tqdm(week_paths):
    # Extract patient name from the path; assumes folder structure .../Patient-XXX/week-000*
    patient_name = os.path.basename(os.path.dirname(week_folder))
    # Create a dedicated output folder for the patient
    output_dir = os.path.join(output_base, patient_name)
    os.makedirs(output_dir, exist_ok=True)
    
    # Determine the T2 mask path.
    # Option 2: HD-GLIO-AUTO-segmentation native mask
    mask_option2 = os.path.join(week_folder, "HD-GLIO-AUTO-segmentation", "native", "segmentation_T2_origspace.nii.gz")
    # Option 1: DeepBraTumIA segmentation mask
    mask_option1 = os.path.join(week_folder, "DeepBraTumIA-segmentation", "native", "segmentation", "t2_seg_mask.nii.gz")
    
    if os.path.exists(mask_option1):
        week_T2_mask_path = mask_option1
    elif os.path.exists(mask_option2):
        week_T2_mask_path = mask_option2
    else:
        print(f"No T2 mask found for {week_folder}. Skipping registration for this folder.")
        continue
    
    print(f"Registering patient {patient_name} for week folder: {week_folder}")
    
    # Call the registration function for one week.
    register_patient_one_week(
        fixed_image_path,
        week_folder,
        week_T2_mask_path,
        output_dir
    )

  0%|                                                    | 0/31 [00:00<?, ?it/s]

Registering patient Patient-037 for week folder: /Users/yifanli/Desktop/dataset/GBM_public/perfect_week_N/Patient-037/week-037

[INFO] Registering T1 for the week...

[INFO] Fixed image header:
  Origin: (120.0, 129.0, -68.0)
  Spacing: (1.0, 1.0, 1.0)
  Direction: (-1.0, 0.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0, 1.0)
  Size: (240, 240, 155)

[INFO] Moving image header:
  Origin: (-100.03621673583984, -128.8906707763672, -10.764339447021484)
  Spacing: (0.6875, 0.6875, 6.0)
  Direction: (0.9994297967377739, 0.012213443062499248, 0.03147877570990693, -0.021489132690932406, 0.949182269652408, 0.31399240760352987, -0.02604416573598793, -0.31448982620438176, 0.9489035012303116)
  Size: (280, 320, 24)

[INFO] Starting rigid registration...
[INFO] Rigid transform parameters: itk::simple::Transform
 CompositeTransform (0x7f9c4e7d05c0)
   RTTI typeinfo:   itk::CompositeTransform<double, 3u>
   Reference Count: 1
   Modified Time: 128060
   Debug: Off
   Object Name: 
   Observers: 
     none
   Tran

[INFO] Final affine transform parameters: itk::simple::Transform
 CompositeTransform (0x7f9bee7a17d0)
   RTTI typeinfo:   itk::CompositeTransform<double, 3u>
   Reference Count: 1
   Modified Time: 136995
   Debug: Off
   Object Name: 
   Observers: 
     none
   Transforms in queue, from begin to end:
   >>>>>>>>>
   AffineTransform (0x7f9c7f9050b0)
     RTTI typeinfo:   itk::AffineTransform<double, 3u>
     Reference Count: 1
     Modified Time: 136986
     Debug: Off
     Object Name: 
     Observers: 
       none
     Matrix: 
       1.00842 0.032203 -0.00880449 
       0.000965066 1.05107 0.128693 
       -0.0100689 -0.180215 1.10646 
     Offset: [-2.29085, -30.8987, 7.98064]
     Center: [-0.107425, 23.2723, 12.0807]
     Translation: [-1.64868, -28.1555, 5.07383]
     Inverse: 
       0.991786 -0.0284657 0.0112029 
       -0.00197628 0.932862 -0.108518 
       0.00870345 0.151681 0.886209 
     Singular: 0
   End of MultiTransform.
<<<<<<<<<<
   TransformsToOptimizeFlags, begin

  3%|█▎                                       | 1/31 [10:06<5:03:28, 606.94s/it]

[INFO] Registration complete. Registered image saved to: /Users/yifanli/Desktop/dataset/GBM_public/procesed_perfect_week_N/Patient-037/registered_FLAIR.nii.gz

[INFO] All registrations for the week are complete.
Registering patient Patient-030 for week folder: /Users/yifanli/Desktop/dataset/GBM_public/perfect_week_N/Patient-030/week-085

[INFO] Registering T1 for the week...

[INFO] Fixed image header:
  Origin: (120.0, 129.0, -68.0)
  Spacing: (1.0, 1.0, 1.0)
  Direction: (-1.0, 0.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0, 1.0)
  Size: (240, 240, 155)

[INFO] Moving image header:
  Origin: (79.5, -133.01205444335938, 119.22891235351562)
  Spacing: (1.0, 1.0, 1.0)
  Direction: (-0.0, 0.0, -1.0, 1.0, -0.0, 0.0, 0.0, -1.0, 0.0)
  Size: (256, 256, 160)

[INFO] Starting rigid registration...
[INFO] Rigid transform parameters: itk::simple::Transform
 CompositeTransform (0x7f9c7f89c990)
   RTTI typeinfo:   itk::CompositeTransform<double, 3u>
   Reference Count: 1
   Modified Time: 142704
   Debug: Of

...

 97%|██████████████████████████████████████▋ | 30/31 [6:09:08<12:50, 770.81s/it]

[INFO] Registration complete. Registered image saved to: /Users/yifanli/Desktop/dataset/GBM_public/procesed_perfect_week_N/Patient-042/registered_FLAIR.nii.gz

[INFO] All registrations for the week are complete.
Registering patient Patient-029 for week folder: /Users/yifanli/Desktop/dataset/GBM_public/perfect_week_N/Patient-029/week-223

[INFO] Registering T1 for the week...

[INFO] Fixed image header:
  Origin: (120.0, 129.0, -68.0)
  Spacing: (1.0, 1.0, 1.0)
  Direction: (-1.0, 0.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0, 1.0)
  Size: (240, 240, 155)

[INFO] Moving image header:
  Origin: (78.24698638916016, -147.00865173339844, 88.3646011352539)
  Spacing: (1.0, 1.0, 1.0)
  Direction: (-0.0, 0.0, -1.0, 1.0, -0.0, 0.0, 0.0, -1.0, 0.0)
  Size: (256, 256, 160)

[INFO] Starting rigid registration...
[INFO] Rigid transform parameters: itk::simple::Transform
 CompositeTransform (0x7f9c4e7d05c0)
   RTTI typeinfo:   itk::CompositeTransform<double, 3u>
   Reference Count: 1
   Modified Time: 566660
 

[INFO] Final affine transform parameters: itk::simple::Transform
 CompositeTransform (0x7f9c3e72bee0)
   RTTI typeinfo:   itk::CompositeTransform<double, 3u>
   Reference Count: 1
   Modified Time: 575688
   Debug: Off
   Object Name: 
   Observers: 
     none
   Transforms in queue, from begin to end:
   >>>>>>>>>
   AffineTransform (0x7f9c7f9ba8c0)
     RTTI typeinfo:   itk::AffineTransform<double, 3u>
     Reference Count: 1
     Modified Time: 575679
     Debug: Off
     Object Name: 
     Observers: 
       none
     Matrix: 
       0.983507 -0.0371205 -0.0599456 
       0.0185673 0.87954 0.200619 
       0.024905 -0.186457 0.94611 
     Offset: [1.57197, -15.9909, 0.599393]
     Center: [-0.107425, 23.2723, 12.0807]
     Translation: [-0.014326, -16.3727, -4.39358]
     Inverse: 
       1.01441 0.0540103 0.0528205 
       -0.0146645 1.08727 -0.23148 
       -0.029593 0.212854 1.00995 
     Singular: 0
   End of MultiTransform.
<<<<<<<<<<
   TransformsToOptimizeFlags, begin() to e

100%|████████████████████████████████████████| 31/31 [6:22:52<00:00, 741.04s/it]

[INFO] Registration complete. Registered image saved to: /Users/yifanli/Desktop/dataset/GBM_public/procesed_perfect_week_N/Patient-029/registered_FLAIR.nii.gz

[INFO] All registrations for the week are complete.
